# N15: The Novel Stacking Architecture## RealMLP (NeurIPS 2024) + TabPFN v3 + GBDT Level-2 Meta-Ensemble**Objective:** Deploy a wholly new novel architecture combining three fundamentally different model families as Level-1 base learners, with a GBDT meta-learner at Level-2.**Architecture:**```Level-1 (3 diverse families, each generating 3-class OOF probabilities):  ├── Family A: RealMLP (Meta-Tuned MLP, no HPO needed, periodic embeddings)  ├── Family B: TabPFN v3 (Zero-shot Transformer, in-context learning)  └── Family C: GBDT Ensemble (CatBoost + LightGBM + HGBC, 3 seeds, TargetEncoder)Level-2 Meta-Learner:  └── LightGBM trained on 9-column L1 OOF probability matrix      (uses SEPARATE fold split to prevent L1/L2 leakage)```**The mechanistic justification:**- Era 1's v0.9 proved RealMLP + HGBC blend reaches 0.95065 (organic best)- Era 2 never used RealMLP at all- TabPFN v3 is genuinely orthogonal to both (in-context learning vs gradient vs backprop)- 3-family stacking should capture error patterns that 2-family blending misses- sklearn TargetEncoder on ALL features (the v0.7 innovation) is used throughout

In [ ]:
!pip install -q tabpfn pytabkit 2>/dev/null || true!pip install -q tabpfn 2>/dev/null || true!pip install -q pytabkit 2>/dev/null || true

In [ ]:
import pandas as pdimport numpy as npimport warningsimport timeimport gcimport torchfrom sklearn.model_selection import StratifiedKFoldfrom sklearn.metrics import balanced_accuracy_scorefrom sklearn.preprocessing import LabelEncoder, TargetEncoderfrom sklearn.impute import SimpleImputerfrom sklearn.ensemble import HistGradientBoostingClassifierfrom sklearn.utils.class_weight import compute_sample_weightimport lightgbm as lgbfrom catboost import CatBoostClassifierwarnings.filterwarnings('ignore')ID_COL, TARGET_COL = "id", "health_condition"CLASSES = ("at-risk", "fit", "unhealthy")N_FOLDS = 5SEED = 42N_CLASSES = 3GPU_ENABLED = torch.cuda.is_available()DEVICE = 'cuda' if GPU_ENABLED else 'cpu'print(f"GPU: {GPU_ENABLED} | Device: {DEVICE}")
import os
# Prefer Kaggle Models-mounted TabPFN-3 weights (avoids PriorLabs browser license flow).
# Add model input: Models → prior-labsai/tabpfn-3 → Add to Notebook.
_KAGGLE_TABPFN = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
if os.path.isdir(_KAGGLE_TABPFN):
    os.environ["TABPFN_MODEL_CACHE_DIR"] = _KAGGLE_TABPFN
    print(f"Using mounted TabPFN weights: {_KAGGLE_TABPFN}")
elif not os.environ.get("TABPFN_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret("TABPFN_TOKEN")
        print("Using TABPFN_TOKEN from Kaggle Secrets")
    except Exception:
        print(
            "WARNING: TabPFN-3 weights not mounted and TABPFN_TOKEN unset. "
            "Add prior-labsai/tabpfn-3 as a notebook input, or set Secret TABPFN_TOKEN."
        )

# Check optional dependenciesTABPFN_OK = Falsetry:    
import os
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
os.environ["TABPFN_NO_BROWSER"] = "1"
TABPFN_CKPT = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1/tabpfn-v3-classifier-v3_default.ckpt"
if not os.path.isfile(TABPFN_CKPT):
    raise FileNotFoundError(
        f"Missing mounted checkpoint: {TABPFN_CKPT}. "
        "Add Models → prior-labsai/tabpfn-3, then Factory reset + Run All."
    )
print(f"TabPFN checkpoint OK: {TABPFN_CKPT}")

from tabpfn import TabPFNClassifier    TABPFN_OK = True    print("TabPFN v3: Available")except ImportError:    print("TabPFN v3: NOT available")REALMLP_OK = Falsetry:    from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier    REALMLP_OK = True    print("RealMLP: Available")except ImportError:    print("RealMLP: NOT available")

In [ ]:
# 1. Load Dataimport osfor p in ['/kaggle/input/competitions/playground-series-s6e7',          '/kaggle/input/playground-series-s6e7', '../../data']:    if os.path.exists(os.path.join(p, 'train.csv')):        DATA_DIR = p; breaktrain_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))print(f"Train: {train_df.shape} | Test: {test_df.shape}")le_target = LabelEncoder()y_train = le_target.fit_transform(train_df[TARGET_COL])cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender',            'smoking_alcohol', 'sleep_quality']num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',            'step_count', 'exercise_duration', 'water_intake']

In [ ]:
# 2. Feature Engineering: sklearn TargetEncoder on ALL featuresall_cols = cat_cols + num_cols# Raw features for TargetEncoder (cast to string)X_train_str = train_df[all_cols].copy()X_test_str = test_df[all_cols].copy()for col in all_cols:    X_train_str[col] = X_train_str[col].fillna('__NA__').astype(str)    X_test_str[col] = X_test_str[col].fillna('__NA__').astype(str)# Raw numeric (imputed)num_imputer = SimpleImputer(strategy='median')X_train_num = num_imputer.fit_transform(train_df[num_cols])X_test_num = num_imputer.transform(test_df[num_cols])# TargetEncoder (cross-fitted, multiclass)te = TargetEncoder(cv=5, target_type='multiclass', random_state=SEED)X_train_te = te.fit_transform(X_train_str, y_train)X_test_te = te.transform(X_test_str)# Combined feature matrixX_train_full = np.hstack([X_train_num, X_train_te])X_test_full = np.hstack([X_test_num, X_test_te])print(f"Feature matrix: {X_train_full.shape[1]} columns")print(f"  Raw numeric: {X_train_num.shape[1]}")print(f"  TE columns: {X_train_te.shape[1]}")# CV splits (shared across all Level-1 models)skf_l1 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)l1_folds = list(skf_l1.split(X_train_full, y_train))

In [ ]:
# 3. LEVEL-1 FAMILY A: RealMLP (NeurIPS 2024)# The mechanistic justification: RealMLP was Era 1's best solo model (0.95062 OOF).# It uses meta-tuned defaults and periodic numeric embeddings — no HPO needed.# Era 2 NEVER used RealMLP.oof_A = np.zeros((len(train_df), N_CLASSES))tst_A = np.zeros((len(test_df), N_CLASSES))if REALMLP_OK:    print("=== Level-1 Family A: RealMLP ===")    t0 = time.time()        for fold, (tr_i, va_i) in enumerate(l1_folds):        print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)                clf = RealMLP_TD_Classifier(device=DEVICE, random_state=SEED+fold, verbosity=0)        clf.fit(            pd.DataFrame(X_train_full[tr_i]),            pd.Series(y_train[tr_i])        )                va_probs = clf.predict_proba(pd.DataFrame(X_train_full[va_i]))        te_probs = clf.predict_proba(pd.DataFrame(X_test_full))                oof_A[va_i] = va_probs        tst_A += te_probs / N_FOLDS                score = balanced_accuracy_score(y_train[va_i], va_probs.argmax(axis=1))        print(f"BAcc={score:.5f}")                del clf; gc.collect()        if GPU_ENABLED: torch.cuda.empty_cache()        score_A = balanced_accuracy_score(y_train, oof_A.argmax(axis=1))    print(f">>> RealMLP OOF: {score_A:.5f} ({time.time()-t0:.0f}s) <<<")else:    print("RealMLP not available. Skipping Family A.")    score_A = 0.0

In [ ]:
# 4. LEVEL-1 FAMILY B: TabPFN v3 (Full Dataset)oof_B = np.zeros((len(train_df), N_CLASSES))tst_B = np.zeros((len(test_df), N_CLASSES))if TABPFN_OK:    print("=== Level-1 Family B: TabPFN v3 ===")    t0 = time.time()        for fold, (tr_i, va_i) in enumerate(l1_folds):        print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)                clf = TabPFNClassifier(device=DEVICE, model_path=TABPFN_CKPT)                try:            clf.fit(X_train_full[tr_i], y_train[tr_i])            va_probs = clf.predict_proba(X_train_full[va_i])            te_probs = clf.predict_proba(X_test_full)        except Exception as e:            print(f"Full fit failed ({e}), using 100K subsample...")            from sklearn.model_selection import train_test_split            sub_idx, _ = train_test_split(                np.arange(len(tr_i)), train_size=min(100000, len(tr_i)),                stratify=y_train[tr_i], random_state=SEED+fold            )            clf = TabPFNClassifier(device=DEVICE, model_path=TABPFN_CKPT)            clf.fit(X_train_full[tr_i[sub_idx]], y_train[tr_i[sub_idx]])            va_probs = clf.predict_proba(X_train_full[va_i])            te_probs = clf.predict_proba(X_test_full)                oof_B[va_i] = va_probs        tst_B += te_probs / N_FOLDS                score = balanced_accuracy_score(y_train[va_i], va_probs.argmax(axis=1))        print(f"BAcc={score:.5f}")                del clf; gc.collect()        if GPU_ENABLED: torch.cuda.empty_cache()        score_B = balanced_accuracy_score(y_train, oof_B.argmax(axis=1))    print(f">>> TabPFN v3 OOF: {score_B:.5f} ({time.time()-t0:.0f}s) <<<")else:    print("TabPFN not available. Skipping Family B.")    score_B = 0.0

In [ ]:
# 5. LEVEL-1 FAMILY C: GBDT Ensemble (CatBoost + LightGBM + HGBC, 3 seeds)print("=== Level-1 Family C: GBDT Ensemble ===")t0 = time.time()oof_C = np.zeros((len(train_df), N_CLASSES))tst_C = np.zeros((len(test_df), N_CLASSES))GBDT_SEEDS = [42, 2026, 999]cb_task = "GPU" if GPU_ENABLED else "CPU"for fold, (tr_i, va_i) in enumerate(l1_folds):    print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)        X_tr, X_va, X_te = X_train_full[tr_i], X_train_full[va_i], X_test_full    y_tr, y_va = y_train[tr_i], y_train[va_i]        sw = compute_sample_weight('balanced', y_tr)    cc = np.bincount(y_tr, minlength=N_CLASSES)    cb_w = [len(y_tr) / (N_CLASSES * c) for c in cc]        fold_va = np.zeros((len(va_i), N_CLASSES))    fold_te = np.zeros((len(test_df), N_CLASSES))    n_models = 3 * len(GBDT_SEEDS)        for seed in GBDT_SEEDS:        # CatBoost        m = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=6,                               class_weights=cb_w, random_seed=seed, verbose=0,                               task_type=cb_task)        m.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100)        fold_va += m.predict_proba(X_va) / n_models        fold_te += m.predict_proba(X_te) / n_models                # LightGBM        lgb_p = dict(n_estimators=1200, learning_rate=0.03, num_leaves=63,                     class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)        if GPU_ENABLED: lgb_p['device_type'] = 'gpu'        m = lgb.LGBMClassifier(**lgb_p)        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],              callbacks=[lgb.early_stopping(100, verbose=False)])        fold_va += m.predict_proba(X_va) / n_models        fold_te += m.predict_proba(X_te) / n_models                # HGBC        m = HistGradientBoostingClassifier(            max_iter=1200, learning_rate=0.03, max_leaf_nodes=63,            class_weight='balanced', random_state=seed,            early_stopping=True, validation_fraction=0.1, n_iter_no_change=100)        m.fit(X_tr, y_tr, sample_weight=sw)        fold_va += m.predict_proba(X_va) / n_models        fold_te += m.predict_proba(X_te) / n_models        oof_C[va_i] = fold_va    tst_C += fold_te / N_FOLDS    print(f"BAcc={balanced_accuracy_score(y_va, fold_va.argmax(axis=1)):.5f}")score_C = balanced_accuracy_score(y_train, oof_C.argmax(axis=1))print(f">>> GBDT Ensemble OOF: {score_C:.5f} ({time.time()-t0:.0f}s) <<<")

In [ ]:
# 6. LEVEL-2: Meta-Learner Stacking# Build the L1 feature matrix from available familiesprint("=== LEVEL-2: Meta-Learner Stacking ===")available_oofs = []available_tsts = []labels = []if REALMLP_OK and score_A > 0.90:    available_oofs.append(oof_A)    available_tsts.append(tst_A)    labels.append("RealMLP")if TABPFN_OK and score_B > 0.90:    available_oofs.append(oof_B)    available_tsts.append(tst_B)    labels.append("TabPFN")# GBDT is always availableavailable_oofs.append(oof_C)available_tsts.append(tst_C)labels.append("GBDT")print(f"Stacking {len(available_oofs)} Level-1 families: {labels}")# Construct L1 OOF matrixL1_oof = np.hstack(available_oofs)L1_tst = np.hstack(available_tsts)print(f"L1 feature matrix: {L1_oof.shape[1]} columns")if len(available_oofs) >= 2:    # Use a SEPARATE fold split for Level-2 (prevents L1/L2 leakage)    skf_l2 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED * 3)        L2_oof = np.zeros((len(train_df), N_CLASSES))    L2_tst = np.zeros((len(test_df), N_CLASSES))    l2_scores = []        for fold, (tr_i, va_i) in enumerate(skf_l2.split(L1_oof, y_train)):        lgb_p = dict(n_estimators=500, learning_rate=0.05, num_leaves=15,                     class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1)        if GPU_ENABLED: lgb_p['device_type'] = 'gpu'                meta = lgb.LGBMClassifier(**lgb_p)        meta.fit(L1_oof[tr_i], y_train[tr_i],                 eval_set=[(L1_oof[va_i], y_train[va_i])],                 callbacks=[lgb.early_stopping(50, verbose=False)])                L2_oof[va_i] = meta.predict_proba(L1_oof[va_i])        L2_tst += meta.predict_proba(L1_tst) / N_FOLDS                s = balanced_accuracy_score(y_train[va_i], L2_oof[va_i].argmax(axis=1))        l2_scores.append(s)        print(f"  L2 Fold {fold+1}: BAcc={s:.5f}")        l2_score = balanced_accuracy_score(y_train, L2_oof.argmax(axis=1))    print(f"\n>>> Level-2 Meta OOF: {l2_score:.5f} <<<")        # Compare L2 stacking vs simple averaging    simple_blend = np.mean(available_oofs, axis=0)    simple_score = balanced_accuracy_score(y_train, simple_blend.argmax(axis=1))    print(f"Simple Average OOF:    {simple_score:.5f}")        # Also try optimal weighted blend    from scipy.optimize import minimize    def neg_bacc(w):        w = np.array(w)        w = np.abs(w) / np.abs(w).sum()        blend = sum(w[i] * available_oofs[i] for i in range(len(w)))        return -balanced_accuracy_score(y_train, blend.argmax(axis=1))        res = minimize(neg_bacc, x0=np.ones(len(available_oofs))/len(available_oofs),                   method='Nelder-Mead', options={'maxiter': 2000})    opt_w = np.abs(res.x) / np.abs(res.x).sum()    opt_blend_oof = sum(opt_w[i] * available_oofs[i] for i in range(len(opt_w)))    opt_blend_score = balanced_accuracy_score(y_train, opt_blend_oof.argmax(axis=1))    opt_blend_tst = sum(opt_w[i] * available_tsts[i] for i in range(len(opt_w)))    print(f"Optimal Blend OOF:     {opt_blend_score:.5f} (weights: {dict(zip(labels, [f'{w:.3f}' for w in opt_w]))})")        # Pick the best approach    best_score = max(l2_score, simple_score, opt_blend_score, score_C)    if best_score == l2_score:        final_probs = L2_tst        final_method = "Level-2 LightGBM Meta-Learner"    elif best_score == opt_blend_score:        final_probs = opt_blend_tst        final_method = f"Optimal Blend ({dict(zip(labels, [f'{w:.3f}' for w in opt_w]))})"    elif best_score == simple_score:        final_probs = np.mean(available_tsts, axis=0)        final_method = "Simple Average"    else:        final_probs = tst_C        final_method = "GBDT Ensemble solo"    final_score = best_scoreelse:    print("Only 1 family available. Using GBDT Ensemble directly.")    final_probs = tst_C    final_score = score_C    final_method = "GBDT Ensemble solo"print(f"\n>>> BEST: {final_method} | OOF: {final_score:.5f} <<<")

In [ ]:
# 7. High-Confidence Pseudo-Labeling EnhancementPSEUDO_THRESHOLD = 0.99max_probs = final_probs.max(axis=1)pseudo_mask = max_probs >= PSEUDO_THRESHOLDn_pseudo = pseudo_mask.sum()print(f"Pseudo-labels available: {n_pseudo} ({n_pseudo/len(test_df)*100:.1f}%)")if n_pseudo > 1000:    pseudo_y = final_probs[pseudo_mask].argmax(axis=1)    aug_X = np.vstack([X_train_full, X_test_full[pseudo_mask]])    aug_y = np.concatenate([y_train, pseudo_y])        # Quick retrain with augmented data    aug_tst = np.zeros((len(test_df), N_CLASSES))    skf_aug = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED*5)        for fold, (tr_i, va_i) in enumerate(skf_aug.split(aug_X, aug_y)):        sw = compute_sample_weight('balanced', aug_y[tr_i])        cc = np.bincount(aug_y[tr_i], minlength=N_CLASSES)        cb_w = [len(aug_y[tr_i]) / (N_CLASSES * c) for c in cc]                fp = np.zeros((len(test_df), N_CLASSES))                m = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=6,                               class_weights=cb_w, random_seed=SEED, verbose=0,                               task_type=cb_task)        m.fit(aug_X[tr_i], aug_y[tr_i])        fp += m.predict_proba(X_test_full) / 3                lgb_p = dict(n_estimators=1200, learning_rate=0.03, num_leaves=63,                     class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1)        if GPU_ENABLED: lgb_p['device_type'] = 'gpu'        m = lgb.LGBMClassifier(**lgb_p)        m.fit(aug_X[tr_i], aug_y[tr_i])        fp += m.predict_proba(X_test_full) / 3                m = HistGradientBoostingClassifier(            max_iter=1200, learning_rate=0.03, max_leaf_nodes=63,            class_weight='balanced', random_state=SEED)        m.fit(aug_X[tr_i], aug_y[tr_i], sample_weight=sw)        fp += m.predict_proba(X_test_full) / 3                aug_tst += fp / N_FOLDS        # Conservative blend    final_probs = 0.7 * final_probs + 0.3 * aug_tst    print("Applied pseudo-labeling (70/30 blend)")else:    print("Too few pseudo-labels, skipping.")

In [ ]:
# 8. Generate Final Submissionfinal_preds = final_probs.argmax(axis=1)sub_df = pd.DataFrame({    ID_COL: test_df[ID_COL].astype("int64"),    TARGET_COL: le_target.inverse_transform(final_preds)})sub_df.to_csv("submission.csv", index=False)print(f"\n{'='*60}")print(f"N15: THE NOVEL STACKING ARCHITECTURE — FINAL RESULTS")print(f"{'='*60}")if REALMLP_OK and score_A > 0.90:    print(f"RealMLP solo:           {score_A:.5f}")if TABPFN_OK and score_B > 0.90:    print(f"TabPFN v3 solo:         {score_B:.5f}")print(f"GBDT Ensemble solo:     {score_C:.5f}")print(f"Final method:           {final_method}")print(f"Final OOF BAcc:         {final_score:.5f}")print(f"{'='*60}")print(f"Previous organic best:  0.95065 (v0.9 RealMLP+HGBC)")print(f"Delta vs v0.9:          {final_score - 0.95065:+.5f}")print(f"\nSubmission: {len(sub_df)} rows")print(sub_df[TARGET_COL].value_counts())